In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

norm_csv = '/home/server/Projects/data/base/normalization_stats.csv'

preop_csv =     "/home/server/Projects/data/AKI/preop_data_test.csv"
intraop_csv =   "/home/server/Projects/data/AKI/feature_engineered.csv"


df_preop = pd.read_csv(preop_csv)
df_intraop = pd.read_csv(intraop_csv)

df = df_preop.merge(df_intraop, on='op_id', how='inner')

combined_csv = '/home/server/Projects/data/base/tabular_combined.csv'
preop_csv = '/home/server/Projects/data/base/tabular_preop.csv'
intraop_csv = '/home/server/Projects/data/base/tabular_intraop.csv'


df.replace([np.inf, -np.inf], np.nan, inplace=True)
cols_to_pop = ["postop_creatinine", "subject_id", 
            "opstart_time", "opend_time", "inhosp_death_time", 
            "allcause_death_time"]
removed = []
for col in cols_to_pop:
    if col in df.columns or "Unnamed" in col:
        removed.append(col)
        df.pop(col)
print(f"Removed columns: {removed}")

# # missing data indicators and zero out missing
# indicator_columns = []
# for col in df.columns:
#     if df[col].isnull().any():
#         # indicates missing, e.g. True => missing
#         indicator_columns.append(pd.Series(df[col].isna(), name=f'{col}_isna'))
# df = pd.concat([df] + indicator_columns, axis=1)
# df.fillna(df.mean(), inplace=True)


Removed columns: ['subject_id', 'opstart_time', 'opend_time', 'inhosp_death_time', 'allcause_death_time']


In [6]:


# replace outliers
int_columns = df.select_dtypes(include=['int']).columns
df[int_columns] = df[int_columns].astype(float)
np.random.seed(42)

# Ignore columns that are not numerical for outlier handling and normalization
ignore = ['op_id', 'age', 'emop', 'num_card_events', 'antype', 'sex', 'asa']
for col in df.columns:
    if ('department' in col) or ('_isna' in col) or ('aki' in col):
        ignore.append(col)

# remove outliers
for col in df.columns:
    if col not in ignore:
        lower_1 = df[col].quantile(0.01)
        upper_1 = df[col].quantile(0.99)
        lower_05 = df[col].quantile(0.005)
        lower_5 = df[col].quantile(0.05)
        upper_95 = df[col].quantile(0.95)
        upper_995 = df[col].quantile(0.995)
        mask_lower = df[col] < lower_1
        mask_upper = df[col] > upper_1
        df.loc[mask_lower, col] = np.random.uniform(lower_05, lower_5, mask_lower.sum())
        df.loc[mask_upper, col] = np.random.uniform(upper_95, upper_995, mask_upper.sum())


# normalize numeric columns
cols_to_norm = [col for col in df.columns if col not in ignore]
scaler = StandardScaler()

scaler.fit(df[cols_to_norm])

df_stats = pd.DataFrame({
    'mean': scaler.mean_,
    'var': scaler.var_
}, index=cols_to_norm)

# Apply normalization
df[cols_to_norm] = scaler.transform(df[cols_to_norm])

preop_cols = []
for col in df_preop.columns:
    for col_2 in df.columns:
        if col in col_2:
            preop_cols.append(col_2)
intraop_cols = []
for col in df_intraop.columns:
    for col_2 in df.columns:
        if col in col_2:
            intraop_cols.append(col_2)



In [7]:

# you need to give imputer.fit_transform the whole df, not just the columns
# thats why you should do the -99 first
from sklearn.impute import KNNImputer
nan_percentage = (df.isna().mean() * 100).to_dict()
# if missing rate is greater than 10%, replace NANs with -99 post normalization
cols = [col for col, nan_pct in nan_percentage.items() if nan_pct >= 10]
print(f'{len(cols)} columns to be simply flagged with -99')
df[cols] = pd.DataFrame(df[cols].fillna(-99), columns=cols, index=df.index)
# if missing rate is less than 10% use knn imputer. 
# takes 40 minutes
cols = [col for col, nan_pct in nan_percentage.items() if nan_pct < 10]
print(f'{len(cols)} columns to be imputed')


185 columns to be simply flagged with -99


90 columns to be imputed


In [8]:
df

,op_id,age,sex,height,weight,asa,emop,BSA,BMI,booking_case_length,...,sum_ebl,sum_ffp,sum_ftn,sum_n2o,sum_pc,sum_pheresis,sum_rbc,sum_uo,fluids_agg,equiv_MAC_totals
0,446345798.0,60.0,True,0.920347,0.198292,4.0,1.0,0.424074,-0.387110,0.820909,...,-0.300723,-99.000000,-99.000000,-99.0,-99.0,-99.0,-0.406649,-0.704110,0.359924,-99.000000
1,415554065.0,55.0,True,1.486026,0.198292,2.0,1.0,0.562692,-0.739825,-0.818673,...,-99.000000,-99.000000,0.072638,-99.0,-99.0,-99.0,-99.000000,-99.000000,-0.481282,-0.327019
2,421885878.0,35.0,True,2.602269,1.925219,2.0,1.0,2.255900,0.265581,-0.464169,...,-99.000000,-99.000000,-0.420080,-99.0,-99.0,-99.0,-99.000000,-99.000000,NaN,-99.000000
3,495921883.0,55.0,False,-1.342371,-0.665171,2.0,1.0,-0.866624,0.156664,0.244839,...,-99.000000,-99.000000,-0.682862,-99.0,-99.0,-99.0,-99.000000,-99.000000,-0.544215,-0.327019
4,430375744.0,35.0,False,-0.776691,0.630024,1.0,1.0,0.337755,1.462960,-0.730047,...,0.714442,-99.000000,-0.567895,-99.0,-99.0,-99.0,-99.000000,-99.000000,-0.103683,-99.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122515,452652040.0,70.0,True,1.486026,0.630024,2.0,0.0,0.926347,-0.285267,-0.730047,...,-99.000000,-99.000000,-99.000000,-99.0,-99.0,-99.0,-99.000000,-99.000000,-0.103683,-99.000000
122516,481612000.0,65.0,False,-1.342371,0.198292,2.0,0.0,-0.151920,1.394072,-0.552795,...,-0.796822,-99.000000,-99.000000,-99.0,-99.0,-99.0,-99.000000,-99.000000,-0.129597,-99.000000
122517,404672299.0,75.0,True,0.354667,0.630024,3.0,0.0,0.636513,0.509479,2.903620,...,-0.310484,0.953296,-99.000000,-99.0,-99.0,-99.0,-0.406649,-1.033799,-0.901596,-99.000000
122518,472189264.0,35.0,True,2.051706,0.630024,1.0,0.0,1.068158,-0.633902,-0.685734,...,-99.000000,-99.000000,-99.000000,-99.0,-99.0,-99.0,-99.000000,-99.000000,0.043161,-99.000000


In [9]:
imputer = KNNImputer(n_neighbors=5)
df = pd.DataFrame(imputer.fit_transform(df), columns=df.columns, index=df.index)



In [10]:
df

,op_id,age,sex,height,weight,asa,emop,BSA,BMI,booking_case_length,...,sum_ebl,sum_ffp,sum_ftn,sum_n2o,sum_pc,sum_pheresis,sum_rbc,sum_uo,fluids_agg,equiv_MAC_totals
0,446345798.0,60.0,1.0,0.920347,0.198292,4.0,1.0,0.424074,-0.387110,0.820909,...,-0.300723,-99.000000,-99.000000,-99.0,-99.0,-99.0,-0.406649,-0.704110,0.359924,-99.000000
1,415554065.0,55.0,1.0,1.486026,0.198292,2.0,1.0,0.562692,-0.739825,-0.818673,...,-99.000000,-99.000000,0.072638,-99.0,-99.0,-99.0,-99.000000,-99.000000,-0.481282,-0.327019
2,421885878.0,35.0,1.0,2.602269,1.925219,2.0,1.0,2.255900,0.265581,-0.464169,...,-99.000000,-99.000000,-0.420080,-99.0,-99.0,-99.0,-99.000000,-99.000000,-0.149071,-99.000000
3,495921883.0,55.0,0.0,-1.342371,-0.665171,2.0,1.0,-0.866624,0.156664,0.244839,...,-99.000000,-99.000000,-0.682862,-99.0,-99.0,-99.0,-99.000000,-99.000000,-0.544215,-0.327019
4,430375744.0,35.0,0.0,-0.776691,0.630024,1.0,1.0,0.337755,1.462960,-0.730047,...,0.714442,-99.000000,-0.567895,-99.0,-99.0,-99.0,-99.000000,-99.000000,-0.103683,-99.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122515,452652040.0,70.0,1.0,1.486026,0.630024,2.0,0.0,0.926347,-0.285267,-0.730047,...,-99.000000,-99.000000,-99.000000,-99.0,-99.0,-99.0,-99.000000,-99.000000,-0.103683,-99.000000
122516,481612000.0,65.0,0.0,-1.342371,0.198292,2.0,0.0,-0.151920,1.394072,-0.552795,...,-0.796822,-99.000000,-99.000000,-99.0,-99.0,-99.0,-99.000000,-99.000000,-0.129597,-99.000000
122517,404672299.0,75.0,1.0,0.354667,0.630024,3.0,0.0,0.636513,0.509479,2.903620,...,-0.310484,0.953296,-99.000000,-99.0,-99.0,-99.0,-0.406649,-1.033799,-0.901596,-99.000000
122518,472189264.0,35.0,1.0,2.051706,0.630024,1.0,0.0,1.068158,-0.633902,-0.685734,...,-99.000000,-99.000000,-99.000000,-99.0,-99.0,-99.0,-99.000000,-99.000000,0.043161,-99.000000


In [11]:

print(f'saving output to {combined_csv}')
df.to_csv(combined_csv, index=False)
print(f'saving output to {preop_csv}')
df[preop_cols].to_csv(preop_csv, index=False)
print(f'saving output to {intraop_csv}')
df[intraop_cols].to_csv(intraop_csv, index=False)

print(f'saving normalization stats to {norm_csv}')
df_stats.to_csv(norm_csv)

saving output to /home/server/Projects/data/base/tabular_combined.csv
saving output to /home/server/Projects/data/base/tabular_preop.csv
saving output to /home/server/Projects/data/base/tabular_intraop.csv
saving normalization stats to /home/server/Projects/data/base/normalization_stats.csv
